## Import Individual Stock Returns and Characteristics

In [1]:
# Packages
import numpy as np
import pandas as pd
from pathlib import Path
from pandas.tseries.offsets import MonthEnd
import wrds
#import openassetpricing as oap

# Data directory
datapath = Path('/work/rw196/data/')

# WRDS connection
conn = wrds.Connection(wrds_username='adamwang08')
# conn.describe_table(library='contrib', table='global_factor')

Loading library list...
Done


### From JKP

In [2]:
# # downloading and extracting list of developed countries
# countries = pd.read_excel('https://github.com/bkelly-lab/ReplicationCrisis/raw/master/GlobalFactors/Country%20Classification.xlsx')
# countries_rel = countries[countries['msci_development'] == 'developed']['excntry'].tolist()

# downloading and extracting list of characteristics
chars = pd.read_excel('https://github.com/bkelly-lab/ReplicationCrisis/raw/master/GlobalFactors/Factor%20Details.xlsx')
chars_jkp = chars[chars['abr_jkp'].notna()].sort_values(by=['abr_jkp']).reset_index(drop=True)
chars_jkp = chars_jkp[['abr_jkp','abr_hxz','group','name_new','cite','in-sample period','t-stat','p-value','direction','significance']]
#chars_chosen = chars_jkp[chars_jkp['significance']==1].reset_index(drop=True)
chars_chosen = chars_jkp['abr_jkp'].tolist()
len(chars_chosen)

153

JKP applies 4 screens:
- exch_main=1 (only use data from the main exchanges within a country)
- primary_sec=1 (if a firm has multiple securities outstanding, only retain the primary security as identified by Compustat)
- obs_main=1 (if CRSP and Compustat have data for the same stock, only retain the observation from CRSP)
- common=1 (only retrieve common stocks)

However, these filters may exclude a very small number of SP500 firms. For this reason, I do not use these filters when focusing on S&P 500 firms. 

#### Import the full dataset.

In [3]:
%%time
# Stock characteristics in USA
sql_query= f"""
            SELECT  excntry, eom, permno, size_grp, me, ret, ret_exc, sic, {', '.join(map(str, chars_chosen))}
                    FROM contrib.global_factor
                    WHERE excntry='USA' and eom >= '1959-12-01' and eom <= '2025-12-31' and common=1 and exch_main=1 and primary_sec=1 and obs_main=1
            """

charc = conn.raw_sql(sql_query)

CPU times: user 3min 14s, sys: 24.7 s, total: 3min 38s
Wall time: 13min 28s


In [5]:
charc = charc.reset_index(drop=True)
charc

,excntry,eom,permno,size_grp,me,ret,ret_exc,sic,age,aliq_at,...,taccruals_at,taccruals_ni,tangibility,tax_gr1a,turnover_126d,turnover_var_126d,z_score,zero_trades_126d,zero_trades_21d,zero_trades_252d
0,USA,1962-01-31,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1.0,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,USA,1962-01-31,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,25.0,<NA>,...,<NA>,<NA>,0.743086,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,USA,1962-01-31,<NA>,mega,372.722332,<NA>,<NA>,<NA>,145.0,0.972656,...,<NA>,<NA>,0.735235,0.011927,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,USA,1962-01-31,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,37.0,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,USA,1962-01-31,<NA>,small,58.4999,<NA>,<NA>,<NA>,73.0,<NA>,...,<NA>,<NA>,0.489611,0.112222,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3998293,USA,2025-08-31,93436.0,mega,1076880.65763,0.083044,0.079239,3711.0,248.0,0.788127,...,-0.066389,-1.359411,0.678298,0.050371,0.034529,0.340925,11.852687,0.000951,0.001418,0.001062
3998294,USA,2025-09-30,93436.0,mega,1478249.28,0.332015,0.328706,3711.0,249.0,0.788127,...,-0.066389,-1.359411,0.678298,0.050371,0.0326,0.357704,11.852687,0.001045,0.001188,0.001102
3998295,USA,2025-10-31,93436.0,mega,1518435.92264,0.026624,0.023356,3711.0,250.0,0.780687,...,-0.0405,-0.885243,0.67445,0.048551,0.029699,0.327992,13.878411,0.001166,0.001178,0.001097
3998296,USA,2025-11-30,93436.0,mega,1430667.55923,-0.057802,-0.060814,3711.0,251.0,0.780687,...,-0.0405,-0.885243,0.67445,0.048551,0.028618,0.343989,13.878411,0.001219,0.001153,0.001123


In [6]:
# Save
charc.to_parquet(datapath / 'JKP_stock_charcs.parquet', engine='pyarrow')

### From CZ

Shortcomings: -  no individual stock return
-  only 35 predicators are available
-  does not include important characteristics such as size, price, or short-term reversal


In [ ]:
# # List available release versions
# oap.list_release()

# # By default, it initializes the data source of most recent release
# openap = oap.OpenAP()

# # Original portfolio names of Chen and Zimmermann and the corresponding download names
# openap.list_port()

# # Download list of predictors
# pred_list = openap.dl_signal_doc('pandas')
# pred_chosen = pred_list[(pred_list['Predictability in OP'].isin(['1_clear'])) & 
#                         (pred_list['Signal Rep Quality'].isin(['1_good']))]

# # Download firm characteristics
# charc = openap.dl_signal('pandas', pred_rel)

# # Save
# charc = charc.drop(['excntry','gvkey','jdate'], axis=1).reset_index(drop=True)
# charc['permno'] = charc['permno'].astype(int)
# charc.to_csv(datapath / 'CZ_stock_charcs.csv')